In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import (
    layers,
    models,
    datasets,
    callbacks,
    losses,
    optimizers,
    metrics,
)

# from tensorflow.python.framework.ops import disable_eager_execution
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Conv2DTranspose, Cropping2D, UpSampling2D, Reshape

# disable_eager_execution()

In [ ]:
#def sample_batch(dataset):
#    batch = dataset.take(1).get_single_element()
#    if isinstance(batch, tuple):
#        batch = batch[0]
#    return batch.numpy()


#def display(
#    images, n=10, size=(20, 3), cmap="gray_r", as_type="float32", save_to=None):
#    """
#    Displays n random images from each one of the supplied arrays.
 #   """
  #  if images.max() > 1.0:
   #     images = images / 255.0
    #elif images.min() < 0.0:
     #   images = (images + 1.0) / 2.0

   # plt.figure(figsize=size)
    #for i in range(n):
     #   _ = plt.subplot(1, n, i + 1)
      #  plt.imshow(images[i].astype(as_type), cmap=cmap)
       # plt.axis("off")

    #if save_to:
     #   plt.savefig(save_to)
      #  print(f"\nSaved to {save_to}")

   # plt.show()

# Designing and Building Variational Autoencoder

creating  a new Sampling layer for sampling from the distribution defined by z_mean and z_log_var.

In [ ]:
#Sampling Layer

class Sampling(layers.Layer):
    #We create a new layer by subclassing the keras base Layer
    def call(self, inputs):
        #
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
input_shape = (400,320, 3)

In [ ]:
# Encoder

image_size = 32
embed_dim = 512
encoder_input = layers.Input(shape=input_shape, name="encoder_input")
x =  Conv2D(128, (3, 3), activation='relu', padding='same')(encoder_input)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)

shape_before_flattening = K.int_shape(x)[1:]  # the decoder will need this!

x = layers.Flatten()(x)

#Instead of connecting the Flatten layer directly to the 2D latent space,
#we will connect it to layers z_mean and z_log_var.
z_mean = layers.Dense(embed_dim, name="z_mean")(x)
z_log_var = layers.Dense(embed_dim, name="z_log_var")(x)



#The Sampling layer samples a point z in the latent space from the
#normal distribution defined by the parameters z_mean and z_log_var.
z = Sampling()([z_mean, z_log_var])

#The Keras Model that defines the encoder—a model that takes an input
#image and outputs z_mean, z_log_var and a sampled point z from the
#normal distribution defined by these parameters.
encoder = models.Model(encoder_input, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

In [ ]:
target_shape = (8, 10, 64)
# Decoder
decoder_input = layers.Input(shape=(embed_dim,), name="decoder_input")

x = Dense(target_shape[0] * target_shape[1] * target_shape[2], activation='relu')(decoder_input)
x = Reshape(target_shape)(x)

x = Conv2DTranspose(256, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(16, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2D(3,(3, 3),activation="sigmoid",padding="same")(x)
decoder_output = Cropping2D(cropping=((56,56), (160,160)),name="decoder_output")(x)

decoder = models.Model(decoder_input, decoder_output)
decoder.summary()























# To Train Variational Autoencoder

In [ ]:
#To create the VAE model, we extend the Keras Model class, enabling us to incorporate the KL
#divergence calculation
#into a custom train_step method.

In [ ]:
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs):
        """Call the model on a particular input."""
        z_mean, z_log_var, z = encoder(inputs)
        reconstruction = decoder(z)
        return z_mean, z_log_var, reconstruction

    def train_step(self, data):
        """Step run during training."""
        with tf.GradientTape() as tape:
            #TensorFlow's Gradient Tape helps calculate gradients during a forward pass.
            #To use it, we need wrap the code that performs the operations you want to differentiate within a tf.GradientTape() context.
            #After recording the operations, we can compute the gradient of the loss function concerning certain variables using tape.gradient().
            #These gradients are then used to update the variables with the optimizer.

            z_mean, z_log_var, reconstruction = self(data)
            beta = 500
            reconstruction_loss = tf.reduce_mean(
                beta
                * losses.binary_crossentropy(
                    data, reconstruction, axis=(1, 2, 3)
                )
            )
            kl_loss = tf.reduce_mean(
                tf.reduce_sum(
                    -0.5
                    * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                    axis=1,
                )
            )
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        """Step run during validation."""
        if isinstance(data, tuple):
            data = data[0]

        z_mean, z_log_var, reconstruction = self(data)
        beta = 500
        reconstruction_loss = tf.reduce_mean(
            beta
            * losses.binary_crossentropy(data, reconstruction, axis=(1, 2, 3))
        )
        kl_loss = tf.reduce_mean(
            tf.reduce_sum(
                -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
                axis=1,
            )
        )
        total_loss = reconstruction_loss + kl_loss

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

In [ ]:
vae = VAE(encoder, decoder)

#  Training 

In [ ]:
# Compile the variational autoencoder
optimizer = optimizers.Adam(learning_rate=0.0005)
vae.compile(optimizer=optimizer)

In [ ]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint",
    save_weights_only=False,
    save_freq="epoch",
    monitor="loss",
    mode="min",
    save_best_only=True,
    verbose=0,
)
tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255)
    
path_train = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train"
path_val = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/validation"
    
batch_size = 32
target_size = (input_shape[0],input_shape[1])
train_generator = datagen.flow_from_directory(path_train, target_size=target_size,  batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

validation_generator = datagen.flow_from_directory(path_val, target_size=target_size , batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

In [ ]:
epochs = 2000
batch_size = 32

vae.fit(
    train_generator,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    validation_data=validation_generator,
    callbacks=[model_checkpoint_callback, tensorboard_callback],
)

## Saving Trained Checkpoints

In [ ]:
# Save the final models
vae.save("./models/vae_in5_left")
encoder.save("./models/encoder_in5_left")
decoder.save("./models/decoder_in5_left")

In [ ]:
train_generator[1].shape

# Lets try to reconstruct using our trained variational autoencoder

In [ ]:
import glob, os, cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

in3 = []
in5 = []
outlet = []


path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = img_to_array(img)
        img = img/255.0
        
        if label == "Inlet (3 marking)":
            in3.append(img)

            
        elif label == "Inlet (5 marking)":
            in5.append(img)

            
        elif label == "Outlet":
            outlet.append(img)
print('Done')
in3 = np.array(in3)
in5 = np.array(in5)
outlet =  np.array(outlet)

In [ ]:
val_in3 = []
val_in5 = []
val_outlet = []
val_others= []
 

path = "/path/to/private-data/Croped_Pictures/validation/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = load_img(image_path)
        img = img_to_array(img)
        img = img/255.0
        
        if label == "Inlet (3 marking)":
            val_in3.append(img)

        elif label == "Inlet (5 marking)":
            val_in5.append(img)
            
        elif label == "Outlet":
             val_outlet.append(img)
            
        elif label == "Others":
            val_others.append(img)
print('Done')   
val_in3 = np.array(val_in3)
val_in5 = np.array(val_in5)
val_outlet = np.array(val_outlet)
val_others= np.array(val_others)

In [ ]:
# Create autoencoder predictions and display
z_mean, z_log_var, in3_rec = vae.predict(in3)
z_mean, z_log_var, in5_rec = vae.predict(in5)
z_mean, z_log_var, outlet_rec = vae.predict(outlet)


# Create autoencoder predictions and display
z_mean, z_log_var, val_in3_rec = vae.predict(val_in3)
z_mean, z_log_var, val_in5_rec = vae.predict(val_in5)
z_mean, z_log_var, val_outlet_rec = vae.predict(val_outlet)
z_mean, z_log_var, val_others_rec = vae.predict(val_others)







print("Reconstructions")
# display(reconstructions)

In [ ]:
plt.imshow(val_in3[15])

In [ ]:
plt.imshow(val_in3_rec[15])


In [ ]:
# Function to calculate MSE
def calculate_mse(original, reconstructed):
    return np.mean((original - reconstructed) ** 2)

# Array to store MSE for each image
in5_mse_scores = np.zeros(in5.shape[0])
in3_mse_scores = np.zeros(in3.shape[0])
val_in5_mse_scores = np.zeros(val_in5.shape[0])
outlet_mse_scores = np.zeros(outlet.shape[0])
val_others_mse_scores = np.zeros(val_others.shape[0])

# Calculate MSE for each image
for i in range(in5.shape[0]):
   in5_mse_scores[i] = calculate_mse(in5[i], in5_rec[i])
   
for i in range(in3.shape[0]):
   in3_mse_scores[i] = calculate_mse(in3[i], in3_rec[i])

for i in range(outlet.shape[0]):
   outlet_mse_scores[i] = calculate_mse(outlet[i], outlet_rec[i])

for i in range(val_in5.shape[0]):
   val_in5_mse_scores[i] = calculate_mse(val_in5[i], val_in5_rec[i])

for i in range(val_others.shape[0]):
   val_others_mse_scores[i] = calculate_mse(val_others[i], val_others_rec[i])


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Function to calculate cosine similarity
def calculate_cosine_similarity(original, reconstructed):
    # Flatten the image matrices into vectors
    original_flat = original.flatten().reshape(1, -1)
    reconstructed_flat = reconstructed.flatten().reshape(1, -1)
    # Compute cosine similarity (returns a matrix, get the single value)
    return cosine_similarity(original_flat, reconstructed_flat)[0, 0]

# Assuming the arrays 'in5', 'in3', etc., and their reconstructed counterparts
cosine_scores_in5 = np.zeros(in5.shape[0])
cosine_scores_in3 = np.zeros(in3.shape[0])
cosine_scores_outlet = np.zeros(outlet.shape[0])
cosine_scores_val_in5 = np.zeros(val_in5.shape[0])
cosine_scores_val_others = np.zeros(val_others.shape[0])

# Calculate cosine similarity for each image
for i in range(in5.shape[0]):
   cosine_scores_in5[i] = calculate_cosine_similarity(in5[i], in5_rec[i])

for i in range(in3.shape[0]):
   cosine_scores_in3[i] = calculate_cosine_similarity(in3[i], in3_rec[i])

for i in range(outlet.shape[0]):
   cosine_scores_outlet[i] = calculate_cosine_similarity(outlet[i], outlet_rec[i])

for i in range(val_in5.shape[0]):
   cosine_scores_val_in5[i] = calculate_cosine_similarity(val_in5[i], val_in5_rec[i])

for i in range(val_others.shape[0]):
   cosine_scores_val_others[i] = calculate_cosine_similarity(val_others[i], val_others_rec[i])

print(cosine_scores_in3)  # Output cosine similarities for verification


In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3_mse_scores, in5_mse_scores,  val_in5_mse_scores, outlet_mse_scores, val_others_mse_scores])

# Setting labels
plt.xlabel('Class')
plt.ylabel('MSE Scores')
plt.title('MSE Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2','Inlet Type-2 (Test)', 'Outlet',  'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[cosine_scores_in3, cosine_scores_in5,  cosine_scores_val_in5, cosine_scores_outlet, cosine_scores_val_others])

# Setting labels
plt.xlabel('Class')
plt.ylabel('cosine_similarity Scores')
plt.title('cosine_similarity Scores with Variational Autoencoders')
plt.xticks([0, 1, 2, 3, 4], ['Inlet type-1', 'Inlet Type-2','Inlet Type-2 (Test)', 'Outlet',  'Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

# visualising the Latent Space

In [ ]:
#z_mean, z_var, z = encoder.predict(example_images)
#p = norm.cdf(z)

In [ ]:
z_mean, z_var, z_in5 = encoder.predict(in5)
p_in5 = norm.cdf(z_in5)

z_mean, z_var, z_in3 = encoder.predict(in3)
p_in3 = norm.cdf(z_in3)

z_mean, z_var, z_outlet = encoder.predict(outlet)
p_outlet = norm.cdf(z_outlet)


z_mean, z_var, val_z_in5 = encoder.predict(val_in5)
val_p_in5 = norm.cdf(val_z_in5)

z_mean, z_var, val_z_in3 = encoder.predict(val_in3)
val_p_in3 = norm.cdf(val_z_in3)

z_mean, z_var, val_z_outlet = encoder.predict(val_outlet)
val_p_outlet = norm.cdf(val_z_outlet)

z_mean, z_var, val_z_others = encoder.predict(val_others)
val_p_others = norm.cdf(val_z_others)


In [ ]:
in5_label = np.zeros(len(in5))
in3_label = np.ones(len(in3))
outlet_label = np.ones(len(outlet))*2

val_in5_label = np.ones(len(val_in5))*3

val_others_label = np.ones(len(val_others))*4



 
label_data=np.concatenate([in5_label,in3_label,outlet_label,val_in5_label,val_others_label])
data = np.vstack([z_in5,z_in3,z_outlet,val_z_in5,val_z_others])



In [ ]:
print(label_data.shape)
print(data.shape)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# It's recommended to use another dimensionality reduction method like PCA to reduce to 50 dimensions first
pca_50 = PCA(n_components=50)
data_50 = pca_50.fit_transform(data)

tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=1000)
tsne_results = tsne.fit_transform(data_50)

plt.figure(figsize=(8, 6))
plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=label_data, cmap='rainbow', alpha=0.8, s=3)
plt.colorbar()
plt.title('t-SNE Reduction to 2D')
plt.show()


In [ ]:
 
import seaborn as sns
import pandas as pd
pca_50 = PCA(n_components=100)
data_50 = pca_50.fit_transform(data)  # Ensure 'data' is defined in your scope

# Apply t-SNE
tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=1000)
tsne_results = tsne.fit_transform(data)

# Create a DataFrame for easier plotting
df_tsne = pd.DataFrame(tsne_results, columns=['tsne-2d-one', 'tsne-2d-two'])
df_tsne['label'] = label_data  # Adding your label data to the DataFrame


# Mapping of numeric labels to descriptive labels
label_map = {
    0: "Inlet Type - 2",
    1: "Inlet Type 1",
    2: "Outlet",
    3: "Inlet Type 1 Test",
    4: "Unmarked"
}

# Apply the mapping to the label column
df_tsne['label'] = df_tsne['label'].map(label_map)

color_palette = {
    "Inlet Type - 2": "#1f77b4",
    "Inlet Type 1": "#ff7f0e",
    "Outlet": "#2ca02c",
    "Inlet Type 1 Test": "#d62728",
    "Unmarked": "#9467bd"
}

# Plot using Seaborn for better aesthetics and automatic legend handling
plt.figure(figsize=(10, 8))
scatter = sns.scatterplot(
    x='tsne-2d-one', y='tsne-2d-two',
    hue='label',  # This column is used to map colors to categories
    palette=color_palette,  # Ensures a broad, visually distinct color spectrum
    data=df_tsne,
    legend='full',  # Ensures the legend is displayed
    alpha=0.8,
    s=50  # Adjusts the size of the points in the scatter plot
)

# Enhance the plot with a title and possibly grid lines for better visual parsing
plt.title('t-SNE Reduction to 2D using Seaborn')
plt.grid(True)  # Optional: Adds a grid for easier visualization
plt.show()

In [ ]:



figsize = 8
fig = plt.figure(figsize=(figsize * 2, figsize))
ax = fig.add_subplot(1, 2, 1)
plot_1 = ax.scatter(
    z[:, 0], z[:, 1], cmap="rainbow", c=example_labels, alpha=0.8, s=3
)
plt.colorbar(plot_1)
ax = fig.add_subplot(1, 2, 2)
plot_2 = ax.scatter(
    p[:, 0], p[:, 1], cmap="rainbow", c=example_labels, alpha=0.8, s=3
)
plt.show()

Firstly, the KL divergence loss term keeps z_mean and z_log_var values close to a standard normal distribution. Secondly, the stochastic encoder leads to a more continuous latent space, reducing the occurrence of poorly formed images.


Finally, when we color-code points in the latent space by clothing type, we observe that there is no bias favoring any specific type. The right-hand plot displays the space converted into p-values, revealing nearly equal representation for each color. It's crucial to emphasize that the model learned clothing distinctions independently, without using labels during training, to minimize reconstruction loss.